In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import trim, lower, col

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving data_transaksi (2).csv to data_transaksi (2).csv


In [ ]:
spark = SparkSession.builder.appName("DataCleaning_Transaksi").getOrCreate()

In [ ]:
df = spark.read.csv("data_transaksi (2).csv", header=True, inferSchema=True)
df.show(5)
df.printSchema()
df.describe().show()

+-------+--------------------+-----+----------+-----------------+--------+---------------+
|   Nama|               Email| Umur|Pendapatan|Tanggal Pembelian|  Produk|Nilai Transaksi|
+-------+--------------------+-----+----------+-----------------+--------+---------------+
|Anthony|   dawnday@gmail.com| 24.0|   3553265|       01/13/1974|  Laptop|        2736049|
| Robert|    pbrown@gmail.com| NULL|   5525753|       03-05-2004|  Tablet|        4996241|
|Melissa| ethan99@clayton.com| NULL|   4911936|       28-04-1970|Handpone|        3169456|
|Timothy|johnsonmegan@gmai...|200.0|3140485039|       03/09/2003|  Laptop|         506622|
|  Kelly|keithwebster(at)e...|200.0|3833611071|       17-01-2022| Tablett|        9969204|
+-------+--------------------+-----+----------+-----------------+--------+---------------+
only showing top 5 rows

root
 |-- Nama: string (nullable = true)
 |-- Email: string (nullable = true)
 |-- Umur: double (nullable = true)
 |-- Pendapatan: long (nullable = true)
 |

In [ ]:
df_clean = df.dropDuplicates()

In [ ]:
print(df_clean.columns)

['Nama', 'Email', 'Umur', 'Pendapatan', 'Tanggal Pembelian', 'Produk', 'Nilai Transaksi']


In [ ]:
df_clean = df_clean.na.drop(subset=["Nama", "Email", "Umur", "Pendapatan", "Produk", "Nilai Transaksi"])

In [ ]:
df_clean = df_clean.fillna({'Email': 'Tidak diketahui'})

In [ ]:
df_clean = df_clean.withColumn("Produk", lower(trim(col("Produk"))))
df_clean = df_clean.withColumn("Nama", lower(trim(col("Nama"))))

In [ ]:
df_clean = df_clean.withColumn("Umur", col("Umur").cast("int"))
df_clean = df_clean.withColumn("Pendapatan", col("Pendapatan").cast("double"))
df_clean = df_clean.withColumn("Nilai Transaksi", col("Nilai Transaksi").cast("double"))

In [ ]:
from pyspark.sql.functions import when, avg

# Ubah yang > 120 jadi null
df_clean = df_clean.withColumn("Umur", when(col("Umur") > 120, None).otherwise(col("Umur")))

# Hitung rata-rata umur (tanpa nilai null)
mean_umur = df_clean.select(avg(col("Umur"))).first()[0]

# Isi nilai null dengan rata-rata
df_clean = df_clean.fillna({"Umur": int(mean_umur)})

In [ ]:
df_clean.show(10)
df_clean.printSchema()

+--------+--------------------+----+-------------+-----------------+----------+---------------+
|    Nama|               Email|Umur|   Pendapatan|Tanggal Pembelian|    Produk|Nilai Transaksi|
+--------+--------------------+----+-------------+-----------------+----------+---------------+
| timothy|johnsonmegan@gmai...|  46|3.140485039E9|       03/09/2003|    laptop|       506622.0|
|    mike|owensrobert@hotma...|  59|3.741985672E9|       11/28/2004|    laptop|      7711290.0|
|madeline|martinezmario@gma...|  46|3.200189219E9|       12/05/1979|smartwatch|      1908029.0|
| raymond|santiagoscott(at)...|  46|  2.8663657E9|       07/18/1993|smartwatch|      6530558.0|
|benjamin|brittanygreen@mar...|  46|4.075982395E9|       17-02-1982|    tablet|      9067792.0|
|  leslie|maysannette@gmail...|  46|    8372468.0|       09/13/1981|smartwatch|      4957082.0|
| cynthia|lonnieandrews@cis...|  49|    5139026.0|       02/10/2000|   tablett|      2393089.0|
|  debbie| allison19@yahoo.com|  41|4.97